# DAN 3 — Evaluacija i vizuelizacija

Nastavljamo sa rezultatima iz Dana 2 (`output/clustering_results.csv` i
`output/all_cluster_labels.pkl`). Ako se nastavlja u novoj sesiji, prvo učitati te
fajlove (ćelija ispod).


In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import os

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

results_df = pd.read_csv('../output/clustering_results.csv')
with open('../output/all_cluster_labels.pkl', 'rb') as f:
    all_labels = pickle.load(f)

print(f"Ucitano {len(results_df)} kombinacija algoritam x skup atributa")
results_df.head()


## 1. Tabela rezultata — top kombinacije po Silhouette koeficijentu

In [ ]:
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

top15 = results_df.sort_values('Silhouette', ascending=False).head(15)
top15.round(3)


In [ ]:
top15.round(3).to_csv('../output/top15_results.csv', index=False)
print("Sacuvano: output/top15_results.csv")


## 2. Poređenje algoritama (agregatno, preko svih skupova atributa)

In [ ]:
agg_by_algo = results_df.groupby('Algorithm').agg(
    Silhouette=('Silhouette', 'mean'),
    DaviesBouldin=('DaviesBouldin', 'mean'),
    CalinskiHarabasz=('CalinskiHarabasz', 'mean'),
    K=('K', 'mean'),
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

agg_sorted = agg_by_algo.sort_values('Silhouette')
axes[0,0].barh(agg_sorted['Algorithm'], agg_sorted['Silhouette'], color='#378ADD')
axes[0,0].set_title('Prosecan Silhouette Score (veci je bolji)')
axes[0,0].set_xlabel('Silhouette')

agg_sorted2 = agg_by_algo.sort_values('DaviesBouldin', ascending=False)
axes[0,1].barh(agg_sorted2['Algorithm'], agg_sorted2['DaviesBouldin'], color='#F2A623')
axes[0,1].set_title('Prosecan Davies-Bouldin (manji je bolji)')
axes[0,1].set_xlabel('Davies-Bouldin')

agg_sorted3 = agg_by_algo.sort_values('CalinskiHarabasz')
axes[1,0].barh(agg_sorted3['Algorithm'], agg_sorted3['CalinskiHarabasz'], color='#639922')
axes[1,0].set_title('Prosecan Calinski-Harabasz (veci je bolji)')
axes[1,0].set_xlabel('Calinski-Harabasz')

agg_sorted4 = agg_by_algo.sort_values('K')
axes[1,1].barh(agg_sorted4['Algorithm'], agg_sorted4['K'], color='#E24B4A')
axes[1,1].set_title('Prosecan broj klastera')
axes[1,1].set_xlabel('Broj klastera')

plt.tight_layout()
plt.savefig('../visualizations/algorithm_comparison.png', dpi=150)
plt.show()


## 3. Heatmap — Silhouette po algoritmu i skupu atributa

In [ ]:
pivot_sil = results_df.pivot_table(index='Algorithm', columns='Dataset', values='Silhouette')

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot_sil, annot=True, fmt='.3f', cmap='RdYlGn', center=0.2, ax=ax,
            cbar_kws={'label': 'Silhouette Score'})
ax.set_title('Heatmap Silhouette koeficijenata po algoritmu i skupu atributa')
plt.tight_layout()
plt.savefig('../visualizations/heatmap_silhouette.png', dpi=150)
plt.show()


---
### 🔵 GIT COMMIT — Dan 3
```bash
git add visualizations/algorithm_comparison.png visualizations/heatmap_silhouette.png output/top15_results.csv
git commit -m "feat: agregatno poredjenje algoritama i heatmap po skupovima atributa"
git push
```
---

## 4. Poređenje pun skup vs. redukovani skupovi atributa

Ovo je eksplicitno tražena analiza iz uputstva predmeta — poređenje modela sa svim
atributima naspram redukovanih skupova.


In [ ]:
dataset_comparison = results_df.groupby('Dataset').agg(
    Silhouette_mean=('Silhouette', 'mean'),
    Silhouette_max=('Silhouette', 'max'),
    DaviesBouldin_mean=('DaviesBouldin', 'mean'),
    CalinskiHarabasz_mean=('CalinskiHarabasz', 'mean'),
).round(3).reset_index()

dataset_comparison


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(dataset_comparison))
ax.bar(x - 0.2, dataset_comparison['Silhouette_mean'], width=0.4, label='Prosecan Silhouette', color='#378ADD')
ax.bar(x + 0.2, dataset_comparison['Silhouette_max'], width=0.4, label='Max Silhouette', color='#D85A30')
ax.set_xticks(x)
ax.set_xticklabels(dataset_comparison['Dataset'])
ax.set_ylabel('Silhouette Score')
ax.set_title('Pun skup (Full) vs. redukovani skupovi atributa (PCA-50, TopVar-200)')
ax.legend()
plt.tight_layout()
plt.savefig('../visualizations/full_vs_reduced.png', dpi=150)
plt.show()


**Zapažanje:** Poređenje pokazuje da li redukcija dimenzionalnosti poboljšava ili
narušava kvalitet klasterovanja u odnosu na pun skup od 10935 atributa — ovo se
diskutuje detaljno u tekstualnom delu rada.


---
### 🔵 GIT COMMIT — Dan 3
```bash
git add visualizations/full_vs_reduced.png
git commit -m "viz: poredjenje punog i redukovanih skupova atributa"
git push
```
---

## 5. Analiza najboljeg modela

Biramo najbolji model na osnovu konsenzusa metrika (prvenstveno Silhouette, uz proveru
da rezultat nije artefakt velikog broja šum-tačaka kod DBSCAN-a).


In [ ]:
# Iskljucujemo DBSCAN rezultate sa previse suma (>50% instanci) iz razmatranja za
# "najbolji upotrebljiv model", jer visok Silhouette kod takvih slucajeva cesto
# potice od izuzetno malog broja preostalih (vrlo slicnih) tacaka, a ne od stvarno
# korisne segmentacije celog skupa
results_usable = results_df[results_df['Noise'] < 0.5 * len(y_true) if 'y_true' in dir() else results_df['Noise'] < 772].copy()

best_row = results_usable.sort_values('Silhouette', ascending=False).iloc[0]
print("Najbolji upotrebljivi model:")
print(best_row)


In [ ]:
best_algo = best_row['Algorithm']
best_dataset = best_row['Dataset']

# Pronalazimo odgovarajuce labele iz all_labels recnika (kljucevi su (dataset, kratko_ime_algoritma))
matching_keys = [k for k in all_labels.keys() if k[0] == best_dataset and k[1] in best_algo]
print("Odgovarajuci kljucevi u all_labels:", matching_keys)
best_labels = all_labels[matching_keys[0]]

print(f"\nNajbolji model: {best_algo} na skupu {best_dataset}")
print(f"Broj klastera: {len(set(best_labels)) - (1 if -1 in best_labels else 0)}")
print(f"Broj sum-tacaka: {(best_labels == -1).sum()}")


In [ ]:
# Vizuelizacija najboljeg modela u PCA 2D prostoru, poredjenje sa originalnim Tissue labelama
from sklearn.decomposition import PCA

df_pca50 = pd.read_csv('../data/data_preprocessed_pca50.csv')
y_true = df_pca50['Tissue'].values
X_pca50 = df_pca50.drop(columns=['Tissue']).values

pca_2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca_2.fit_transform(X_pca50)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc1 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=best_labels, cmap='tab10', s=15, alpha=0.7)
axes[0].set_title(f'Najbolji model: {best_algo}\nSkup: {best_dataset}')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[0], label='Klaster ID')

colors_map = {'Breast': '#D85A30', 'Other': '#378ADD'}
for label, color in colors_map.items():
    mask = y_true == label
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=label, alpha=0.6, s=15)
axes[1].set_title('Originalne Tissue labele\n(za poredjenje)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()

plt.tight_layout()
plt.savefig('../visualizations/best_model_visualization.png', dpi=150)
plt.show()


In [ ]:
# Eksterna evaluacija najboljeg modela - kontingencijska tabela sa Tissue labelom
contingency = pd.crosstab(best_labels, y_true, rownames=['Klaster'], colnames=['Tissue'])
print("Kontingencijska tabela (Klaster vs. Tissue):")
print(contingency)

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
print(f"\nARI (Adjusted Rand Index) vs. Tissue: {adjusted_rand_score(y_true, best_labels):.3f}")
print(f"NMI (Normalized Mutual Information) vs. Tissue: {normalized_mutual_info_score(y_true, best_labels):.3f}")


**Napomena o interpretaciji:** Niske vrednosti ARI/NMI ne znače nužno da je
klasterovanje "loše" — one pokazuju da otkrivena struktura klastera ne prati
originalnu (biološki motivisanu) binarnu podelu tkiva. Klasterovanje otkriva
strukturu koja postoji u podacima nezavisno od te labele; ARI/NMI su ovde
dijagnostičko sredstvo, ne apsolutni kriterijum kvaliteta.


---
### 🔵 GIT COMMIT — Dan 3
```bash
git add visualizations/best_model_visualization.png
git commit -m "feat: analiza i vizuelizacija najboljeg modela, eksterna evaluacija"
git push
```
---

In [ ]:
# Cuvanje finalnog najboljeg modela
best_model_info = {
    'algorithm': best_algo,
    'dataset': best_dataset,
    'labels': best_labels,
    'metrics': best_row.to_dict(),
}
with open('../output/best_model.pkl', 'wb') as f:
    pickle.dump(best_model_info, f)

print("Sacuvano: output/best_model.pkl")
print("\nDAN 3 zavrsen. Spremno za Dan 4 - pisanje rada i finalizacija.")


---
### 🔵 GIT COMMIT — Dan 3
```bash
git add output/best_model.pkl output/clustering_results.csv
git commit -m "docs: cuvanje finalnog modela i rezultata evaluacije"
git push
```
---